# LC 208 — Implement Trie (Prefix Tree)
**Difficulty:** Medium | **Category:** Tries / Prefix Trees
**Pattern:** Trie Node with children dict + end-of-word flag

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> A Trie stores strings character by
character in a tree. Each node has a map of children (one per
letter) and a flag marking whether a complete word ends there.
Insert, search, and prefix-check all run in O(m) where m is
the length of the word — independent of how many words exist.
</div>

## Official Problem Statement

A **trie** (pronounced as "try") or **prefix tree** is a tree data
structure used to efficiently store and retrieve keys in a dataset
of strings.

Implement the `Trie` class:
- `Trie()` — Initializes the trie object.
- `void insert(String word)` — Inserts `word` into the trie.
- `boolean search(String word)` — Returns `true` if `word` is in
  the trie (i.e., was inserted before), and `false` otherwise.
- `boolean startsWith(String prefix)` — Returns `true` if there is
  a previously inserted word that has the prefix `prefix`.

**Constraints:**
- `1 <= word.length, prefix.length <= 2000`
- `word` and `prefix` consist only of lowercase English letters.
- At most `3 * 10^4` calls total to `insert`, `search`, `startsWith`.

## What This Is Actually Asking

Build a data structure that stores words and can answer two questions
fast: "Is this exact word stored?" and "Does any stored word start
with this prefix?"

Think of it like a branching path through the alphabet. Each step
follows one letter. If the path exists, a prefix matches. If the
path ends with a "word ends here" flag, a full word matches.

You cannot use a plain set for `startsWith` — it would be too slow
without sorting tricks. The Trie handles both operations in O(m).

## Walk Through an Example by Hand

```
insert("apple")
  root -> a -> p -> p -> l -> e  [is_end=True]

insert("app")
  root -> a -> p -> p             [is_end=True]  <- flag added here
                        -> l -> e [is_end=True]  <- still exists

search("app")      -> walk a,p,p -> is_end=True  -> True
search("ap")       -> walk a,p   -> is_end=False -> False
startsWith("app")  -> walk a,p,p -> path exists  -> True
startsWith("apt")  -> walk a,p   -> no 't' child -> False
```

## The Picture

After inserting "apple", "app", "apply":

```
root
 └── 'a'
      └── 'p'
           └── 'p'  [is_end=True]  <- "app"
                ├── 'l'
                │    ├── 'e'  [is_end=True]  <- "apple"
                │    └── 'y'  [is_end=True]  <- "apply"
```

Each node is a `TrieNode`:
```
TrieNode
├── children: dict[str, TrieNode]  <- branches per letter
└── is_end:   bool                 <- True if word ends here
```

Shared prefixes share nodes. "app", "apple", "apply" all share
the a -> p -> p path. Only 7 nodes total, not 3 * 5 = 15.

## When To Use This Pattern

- When you see **prefix matching**, think Trie.
- When you see **autocomplete / search suggestions**, think Trie.
- When you need **O(m) lookup** regardless of dictionary size,
  think Trie.
- When multiple strings **share common prefixes** and space matters,
  think Trie.
- When you see **word existence + prefix existence** together,
  think Trie over a hash set.

## The Approach

Create a `TrieNode` with a `children` dict and an `is_end` flag.
The `Trie` holds a `root` node that is always empty.

For `insert`, walk the tree one character at a time. If a child
for that character doesn't exist, create one. After the last
character, set `is_end = True` on that node.

For `search`, walk the same path. If any character is missing,
return `False`. After the last character, return `node.is_end`.

For `startsWith`, same walk — but return `True` if the path
exists at all, ignoring `is_end`.

In [ ]:
# Standard library only — no extra imports needed for Trie
from typing import Optional  # used for type hints in docstrings


In [ ]:
def test_harness(TrieClass):
    """
    Replay sequences of (op, args, expected) tuples.
    op is a method name string; args is a list; expected is
    the return value (None for insert).
    """
    sequences = [
        # --- Test 1: basic insert + search + startsWith ---
        [
            ("insert",     ["apple"],  None),
            ("search",     ["apple"],  True),
            ("search",     ["app"],    False),
            ("startsWith", ["app"],    True),
            ("insert",     ["app"],    None),
            ("search",     ["app"],    True),
            ("startsWith", ["app"],    True),
        ],
        # --- Test 2: word not inserted at all ---
        [
            ("search",     ["hello"],  False),
            ("startsWith", ["hel"],    False),
        ],
        # --- Test 3: multiple words sharing prefix ---
        [
            ("insert",     ["apply"],  None),
            ("insert",     ["apple"],  None),
            ("search",     ["apply"],  True),
            ("search",     ["appl"],   False),
            ("startsWith", ["appl"],   True),
            ("startsWith", ["apz"],    False),
        ],
        # --- Test 4: single character word ---
        [
            ("insert",     ["a"],      None),
            ("search",     ["a"],      True),
            ("startsWith", ["a"],      True),
            ("search",     ["ab"],     False),
        ],
    ]

    passed = 0
    failed = 0

    for t_idx, ops in enumerate(sequences, 1):
        trie = TrieClass()
        seq_ok = True
        for op, args, expected in ops:
            result = getattr(trie, op)(*args)
            if expected is not None and result != expected:
                print(
                    f"  FAILED Test {t_idx}: {op}({args}) "
                    f"=> {result}, expected {expected}"
                )
                seq_ok = False
        if seq_ok:
            print(f"  PASSED Test {t_idx}")
            passed += 1
        else:
            failed += 1

    print(f"\nResults: {passed} passed, {failed} failed "
          f"out of {passed + failed} tests")


In [ ]:
class TrieNode:
    """Single node in the Trie."""
    def __init__(self):
        self.children = {}   # char -> TrieNode
        self.is_end   = False  # True if a word ends at this node


class Trie:
    """
    Implement a Prefix Tree (Trie).

    Methods:
        insert(word)      — Add word to the trie. O(m)
        search(word)      — Return True if exact word exists. O(m)
        startsWith(prefix)— Return True if any word has prefix. O(m)
    where m = length of the word/prefix.
    """

    def __init__(self):
        self.root = TrieNode()
        print("[DEBUG] Trie initialised with empty root node")

    def insert(self, word: str) -> None:
        """
        Walk (or create) one node per character.
        Mark is_end=True at the last character.
        """
        print(f"[DEBUG] insert('{word}')")
        pass  # TODO: implement

    def search(self, word: str) -> bool:
        """
        Walk the trie. Return False if any char is missing.
        At the end return node.is_end.
        """
        print(f"[DEBUG] search('{word}')")
        pass  # TODO: implement

    def startsWith(self, prefix: str) -> bool:
        """
        Walk the trie. Return True if full prefix path exists.
        is_end does NOT matter here.
        """
        print(f"[DEBUG] startsWith('{prefix}')")
        pass  # TODO: implement


In [ ]:
# Uncomment and run when solution is ready
# test_harness(Trie)


## Complexity

| Approach | Time (per op) | Space |
|---|---|---|
| Brute force (list scan) | O(n * m) search | O(n * m) |
| Hash set | O(m) search, no prefix | O(n * m) |
| **Trie (optimal)** | **O(m) all ops** | **O(n * m) worst** |

- m = length of word or prefix
- n = number of words stored
- Trie space is O(n * m) worst case but shared prefixes reduce it
  significantly in practice.

## Real World Connection

At Citi, AWS CloudWatch ingests metric names from 6,000+ endpoints
following structured naming like `citi.payments.latency.p99`. A
Trie over these metric-name segments enables instant prefix lookup:
"show me all metrics under `citi.payments`" without scanning all
6,000 names.

When an ETL pipeline needs to route Lambda events by topic prefix,
a Trie router resolves the target in O(m) regardless of how many
routing rules are registered — far better than iterating a list.

Prophet forecasting jobs are tagged with S3 key prefixes like
`s3://citi-ml/prophet/region/endpoint/`. A Trie across those keys
makes autocomplete in internal dashboards instant and memory-
efficient, since shared path segments collapse into shared nodes.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra